[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_6_stereo_vision/17_stereo_calibration.ipynb)

# Notebook 17 — Stereo Camera Calibration

**Part 6: Stereo Vision** | Estimated time: 60 min

---

Now we implement the full stereo calibration pipeline.  
After this notebook, you will have rectified maps ready to plug into a depth estimation system.

```
Stereo calibration pipeline:

  [Mono calibrate L]  ┐
  [Mono calibrate R]  ┤ → [Stereo calibrate] → [Stereo rectify] → [Build maps] → [Save]
  [Shared obj pts]    ┘
       ← NB 07 tools                                                   ← YOU ARE HERE
```

## Recommended Watch

> Same video as NB 16 — it walks through the exact same 8-step stereo calibration pipeline implemented in this notebook. Skip if you already watched it.

| Title | Link | Duration |
|---|---|---|
| **Stereo Vision Camera Calibration with OpenCV: How to Calibrate your Camera with Python Script** | [▶ Watch](https://www.youtube.com/watch?v=yKypaVl6qQo) | ~30 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-python numpy matplotlib -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

## Learning Objectives

By the end of this notebook you will be able to:

- Run the full 8-step stereo calibration pipeline
- Understand `stereoCalibrate` outputs (R, T, E, F)
- Understand `stereoRectify` outputs (R1, R2, P1, P2, Q, ROI1, ROI2)
- Build undistort+rectify maps with `initUndistortRectifyMap`
- Apply maps with `remap` for real-time rectification
- Save and load the stereo maps for production use

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import json

print(f'OpenCV version: {cv2.__version__}')

def show_side_by_side(left, right, title_l='Left', title_r='Right',
                      figsize=(14, 6), cmap=None):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, img, title in [(axes[0], left, title_l), (axes[1], right, title_r)]:
        if cmap:
            ax.imshow(img, cmap=cmap)
        elif len(img.shape) == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 1. Pipeline Overview

```
STEP 1: Capture calibration image pairs
          Left image + Right image of same chessboard
          (3–10 pairs, different angles/positions)

STEP 2: Find chessboard corners in each image (same as mono calibration)

STEP 3: Calibrate LEFT camera individually → K_L, dist_L

STEP 4: Calibrate RIGHT camera individually → K_R, dist_R

STEP 5: stereoCalibrate → R, T (rotation + translation between cameras)
         (Uses CALIB_FIX_INTRINSIC to fix K_L, K_R from steps 3-4)

STEP 6: stereoRectify → R1, R2, P1, P2, Q, ROI1, ROI2

STEP 7: initUndistortRectifyMap → map1_L, map2_L, map1_R, map2_R

STEP 8: Save maps → load in production app, remap() every frame
```

**Why calibrate each camera individually first?**  
Individual calibration gives better intrinsics. Stereo calibration then fixes those intrinsics  
and only estimates the **relative geometry** (R, T) between cameras — a much simpler problem.

## Step 1 & 2: Calibration Data

We generate synthetic stereo calibration data. In production, you would capture real image pairs.

In [ ]:
# ============================================================
# Stereo ground truth parameters
# ============================================================

# Left camera intrinsics ("true" values we'll recover)
K_L_true = np.array([
    [700, 0, 320],
    [0, 700, 240],
    [0, 0, 1]
], dtype=np.float64)

# Right camera intrinsics (slightly different — real cameras always differ)
K_R_true = np.array([
    [698, 0, 318],
    [0, 698, 241],
    [0, 0, 1]
], dtype=np.float64)

# Distortion (small values — reasonably good cameras)
dist_L_true = np.array([-0.15, 0.05, 0.001, -0.001, 0.0])
dist_R_true = np.array([-0.14, 0.04, 0.0005, -0.0008, 0.0])

# Stereo geometry: right camera is 12cm to the right, no rotation
BASELINE = 0.12  # 12 cm
R_true = np.eye(3)  # No rotation between cameras (ideal side-by-side mount)
T_true = np.array([[BASELINE], [0.0], [0.0]])  # Translate by B along X

# Image size
IMG_W, IMG_H = 640, 480

print(f'Ground truth stereo setup:')
print(f'  Baseline: {BASELINE*100:.0f} cm')
print(f'  Left fx: {K_L_true[0,0]:.0f}px  Right fx: {K_R_true[0,0]:.0f}px')
print(f'  Image: {IMG_W}×{IMG_H}px')

In [ ]:
# Generate synthetic stereo calibration image pairs
CHESSBOARD_SIZE = (9, 6)  # inner corners
SQUARE_SIZE = 0.025       # 2.5 cm squares
N_CALIB_PAIRS = 8

# 3D object points for chessboard in Z=0 plane
objp = np.zeros((CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE

def project_with_noise(K, dist, R, t, pts_3d, noise_std=0.3):
    """Project 3D points to 2D with Gaussian noise."""
    pts_2d, _ = cv2.projectPoints(pts_3d, R, t, K, dist)
    pts_2d = pts_2d.reshape(-1, 2)
    if noise_std > 0:
        pts_2d += np.random.normal(0, noise_std, pts_2d.shape)
    return pts_2d.astype(np.float32)

# Camera positions to place the chessboard at
board_poses = [
    (np.array([[-0.1, 0.05, 0.6]]),   np.array([[ 0.15, -0.1,  0.0]])),
    (np.array([[ 0.0, 0.00, 0.55]]),  np.array([[-0.05,  0.1, -0.05]])),
    (np.array([[ 0.1,-0.05, 0.5]]),   np.array([[ 0.0,   0.0,  0.1]])),
    (np.array([[-0.05,0.1,  0.65]]),  np.array([[-0.1,  -0.05, 0.0]])),
    (np.array([[ 0.05,0.0,  0.7]]),   np.array([[ 0.1,   0.05,-0.05]])),
    (np.array([[ 0.0, 0.05, 0.45]]),  np.array([[ 0.05, -0.1,  0.1]])),
    (np.array([[-0.1, 0.0,  0.6]]),   np.array([[-0.15,  0.0,  0.0]])),
    (np.array([[ 0.08,-0.08,0.55]]),  np.array([[ 0.0,   0.1, -0.05]])),
]

all_objpts   = []  # same for both cameras
imgpts_L     = []
imgpts_R     = []

np.random.seed(42)

for tvec_board, rvec_board in board_poses[:N_CALIB_PAIRS]:
    R_board, _ = cv2.Rodrigues(rvec_board.flatten())
    t_board    = tvec_board.flatten().reshape(3, 1)
    
    # Project into left camera
    pts_L = project_with_noise(K_L_true, dist_L_true, R_board, t_board, objp)
    
    # Project into right camera (board is shifted by baseline)
    # Right camera extrinsics = R_true @ R_board, T_true + R_true @ t_board
    R_board_R = R_true @ R_board
    t_board_R = R_true @ t_board + T_true
    pts_R = project_with_noise(K_R_true, dist_R_true, R_board_R, t_board_R, objp)
    
    # Check all points are within image bounds
    def in_bounds(pts, w, h):
        return (pts[:, 0] >= 0).all() and (pts[:, 0] < w).all() and \
               (pts[:, 1] >= 0).all() and (pts[:, 1] < h).all()
    
    if in_bounds(pts_L, IMG_W, IMG_H) and in_bounds(pts_R, IMG_W, IMG_H):
        all_objpts.append(objp)
        imgpts_L.append(pts_L.reshape(-1, 1, 2))
        imgpts_R.append(pts_R.reshape(-1, 1, 2))

print(f'Generated {len(all_objpts)} valid calibration pairs')
print(f'Points per image: {objp.shape[0]} ({CHESSBOARD_SIZE[0]}×{CHESSBOARD_SIZE[1]} grid)')

## Steps 3 & 4: Individual Camera Calibration

Before running stereo calibration, each camera is calibrated individually using the standard chessboard process from NB 07. This gives us an initial K and distortion for each lens. `stereoCalibrate` will refine these further using joint optimization across both cameras, but good individual calibrations give it a better starting point — and if individual reprojection errors are already poor (> 1.0 px), stereo calibration won't fix that.

In [ ]:
frame_size = (IMG_W, IMG_H)

# Calibrate LEFT camera
ret_L, K_L, dist_L, rvecs_L, tvecs_L = cv2.calibrateCamera(
    all_objpts, imgpts_L, frame_size,
    None, None
)

# Calibrate RIGHT camera
ret_R, K_R, dist_R, rvecs_R, tvecs_R = cv2.calibrateCamera(
    all_objpts, imgpts_R, frame_size,
    None, None
)

print('=== Individual Camera Calibration ===')
print()
print('LEFT camera:')
print(f'  RMS reprojection error: {ret_L:.4f} px')
print(f'  fx={K_L[0,0]:.2f} (true: {K_L_true[0,0]:.0f})')
print(f'  cx={K_L[0,2]:.2f} (true: {K_L_true[0,2]:.0f})')
print()
print('RIGHT camera:')
print(f'  RMS reprojection error: {ret_R:.4f} px')
print(f'  fx={K_R[0,0]:.2f} (true: {K_R_true[0,0]:.0f})')
print(f'  cx={K_R[0,2]:.2f} (true: {K_R_true[0,2]:.0f})')

## Step 5: `stereoCalibrate`

Now we estimate the **geometric relationship** between the two cameras.

```python
ret, K_L, dist_L, K_R, dist_R, R, T, E, F = cv2.stereoCalibrate(
    objectPoints,   # list of 3D point arrays (same for both cameras)
    imagePoints1,   # list of 2D corners in left images
    imagePoints2,   # list of 2D corners in right images
    cameraMatrix1,  # K_L (initial estimate)
    distCoeffs1,    # dist_L (initial estimate)
    cameraMatrix2,  # K_R
    distCoeffs2,    # dist_R
    imageSize,      # (W, H)
    flags=cv2.CALIB_FIX_INTRINSIC  # don't re-optimize K and dist
)
```

Returns:
- `R` = 3×3 rotation matrix from left to right camera
- `T` = 3×1 translation vector from left to right camera (in meters if obj pts in meters)
- `E` = Essential matrix (3×3)
- `F` = Fundamental matrix (3×3)

In [ ]:
ret_stereo, K_L_s, dist_L_s, K_R_s, dist_R_s, R, T, E, F = cv2.stereoCalibrate(
    all_objpts,
    imgpts_L,
    imgpts_R,
    K_L.copy(), dist_L.copy(),
    K_R.copy(), dist_R.copy(),
    frame_size,
    criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 200, 1e-8),
    flags=cv2.CALIB_FIX_INTRINSIC
)

print('=== Stereo Calibration Results ===')
print(f'RMS stereo reprojection error: {ret_stereo:.4f} px')
print()
print('Rotation between cameras (R):')
print(R.round(5))
print('  (should be close to Identity — our cameras are parallel)')
print()
print('Translation between cameras (T) in meters:')
print(T.flatten().round(5))
print(f'  Estimated baseline: {np.linalg.norm(T)*100:.2f} cm')
print(f'  True baseline:      {BASELINE*100:.2f} cm')
print(f'  Error: {abs(np.linalg.norm(T) - BASELINE)*100:.2f} cm')

## Step 6: `stereoRectify`

Computes the rectification transforms that make epipolar lines horizontal.

```python
R1, R2, P1, P2, Q, ROI1, ROI2 = cv2.stereoRectify(
    cameraMatrix1, distCoeffs1,  # left camera
    cameraMatrix2, distCoeffs2,  # right camera
    imageSize,
    R, T,                        # from stereoCalibrate
    alpha=0                      # 0=crop to valid region, 1=show full image
)
```

| Output | Meaning |
|---|---|
| `R1` | 3×3 rectification rotation for left camera |
| `R2` | 3×3 rectification rotation for right camera |
| `P1` | 3×4 projection matrix for left (in rectified space) |
| `P2` | 3×4 projection matrix for right (in rectified space) |
| `Q` | 4×4 disparity-to-depth mapping matrix |
| `ROI1`, `ROI2` | Valid pixel regions after rectification |

In [ ]:
# alpha=0: crop output to only valid (non-black) pixels
# alpha=1: keep full image extent (some black borders appear)
ALPHA = 0  # recommendation: use 0 for robotics (clean, no black borders)

R1, R2, P1, P2, Q, ROI1, ROI2 = cv2.stereoRectify(
    K_L_s, dist_L_s,
    K_R_s, dist_R_s,
    frame_size,
    R, T,
    alpha=ALPHA,
    newImageSize=frame_size
)

print('=== Stereo Rectification Results ===')
print()
print('Q matrix (disparity-to-depth):')
print(Q.round(4))
print()
print('P1 (left projection in rectified space):')
print(P1.round(2))
print()
print('Focal length from P1:', P1[0,0])
print('Focal length from P2:', P2[0,0])
print('Baseline from P2:',  -P2[0,3] / P2[0,0], 'm')
print()
print(f'ROI left:  {ROI1}  (valid pixel area after rectification)')
print(f'ROI right: {ROI2}')

## Step 7: Building Undistort+Rectify Maps

`initUndistortRectifyMap` precomputes the pixel-level mapping from a rectified image coordinate  
back to the raw, distorted image coordinate. This is done **once** during setup.

Then in the application loop, `remap` applies the precomputed maps — very fast.

```
Rectified pixel (u_rect, v_rect)
         ↓  (maps encode this transform)
         ↓  undo rectification + undo distortion
         ↓
Raw image pixel (u_raw, v_raw)
         ↓
         → look up color in raw image
```

In [ ]:
# Build maps — this is done ONCE during setup
# map1 = x-coordinate map, map2 = y-coordinate map

map1_L, map2_L = cv2.initUndistortRectifyMap(
    K_L_s, dist_L_s,   # left camera intrinsics
    R1,                 # left rectification rotation
    P1,                 # left projection (in rectified space)
    frame_size,         # output image size
    cv2.CV_32FC1        # map data type
)

map1_R, map2_R = cv2.initUndistortRectifyMap(
    K_R_s, dist_R_s,
    R2,
    P2,
    frame_size,
    cv2.CV_32FC1
)

print('Maps computed:')
print(f'  Left:  map1_L shape={map1_L.shape}, map2_L shape={map2_L.shape}')
print(f'  Right: map1_R shape={map1_R.shape}, map2_R shape={map2_R.shape}')
print()
print('Usage at runtime:')
print('  left_rect  = cv2.remap(left_raw,  map1_L, map2_L, cv2.INTER_LINEAR)')
print('  right_rect = cv2.remap(right_raw, map1_R, map2_R, cv2.INTER_LINEAR)')

In [ ]:
# Demonstrate what remap does on a synthetic chessboard image
def make_distorted_chessboard(K, dist, w=640, h=480, sq=50):
    """Create a synthetic chessboard image with distortion."""
    # Create ideal chessboard
    chess = np.zeros((h, w), dtype=np.uint8)
    for row in range(0, h, sq):
        for col in range(0, w, sq):
            if ((row // sq) + (col // sq)) % 2 == 0:
                chess[row:row+sq, col:col+sq] = 255
    chess_bgr = cv2.cvtColor(chess, cv2.COLOR_GRAY2BGR)
    
    # Apply distortion by inverse-mapping
    # Generate distorted-to-undistorted map
    map_ud1, map_ud2 = cv2.initUndistortRectifyMap(
        K, -dist, None, K, (w, h), cv2.CV_32FC1)  # negative dist = apply distortion
    distorted = cv2.remap(chess_bgr, map_ud1, map_ud2, cv2.INTER_LINEAR)
    return chess_bgr, distorted

# Create distorted image
ideal_L, distorted_L = make_distorted_chessboard(K_L_s, dist_L_s)

# Apply rectification maps
rectified_L = cv2.remap(distorted_L, map1_L, map2_L, cv2.INTER_LINEAR)

# Draw horizontal lines to verify rectification quality
def draw_scanlines(img, step=40, color=(255, 0, 0)):
    out = img.copy()
    h = out.shape[0]
    for y in range(step, h, step):
        cv2.line(out, (0, y), (out.shape[1], y), color, 1)
    return out

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(cv2.cvtColor(draw_scanlines(distorted_L), cv2.COLOR_BGR2RGB))
axes[0].set_title('Distorted input (red = scanlines)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(draw_scanlines(rectified_L), cv2.COLOR_BGR2RGB))
axes[1].set_title('After remap() — rectified\n(scanlines align with chessboard rows)', fontsize=11)
axes[1].axis('off')

# Show map1_L to understand what remap does
im = axes[2].imshow(map1_L, cmap='viridis')
axes[2].set_title('map1_L — X coordinate map\n(encodes the pixel lookup)', fontsize=11)
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, label='source X pixel')

plt.suptitle('cv2.remap() applies precomputed undistort+rectify maps', fontsize=13)
plt.tight_layout()
plt.show()

## Step 8: Saving Stereo Calibration

**Save once, use everywhere.** After calibration, save:
1. The intrinsics and stereo parameters (for reference/debugging)
2. The rectification maps (for fast runtime application)

The video notes recommend saving to XML using `cv2.FileStorage`.  
We also support `.npz` and `.json` for Python workflows.

In [ ]:
CALIB_DIR = '../assets/calibration'
os.makedirs(CALIB_DIR, exist_ok=True)

# ---- Option A: Save maps to .npz (Python-native, fastest to load) ----
npz_path = os.path.join(CALIB_DIR, 'stereo_maps.npz')
np.savez(npz_path,
    map1_L=map1_L, map2_L=map2_L,
    map1_R=map1_R, map2_R=map2_R,
    Q=Q, R=R, T=T,
    K_L=K_L_s, dist_L=dist_L_s,
    K_R=K_R_s, dist_R=dist_R_s,
    R1=R1, R2=R2, P1=P1, P2=P2
)
print(f'Saved stereo maps to: {npz_path}')
file_kb = os.path.getsize(npz_path) / 1024
print(f'File size: {file_kb:.0f} KB')

# ---- Option B: Save parameters to JSON (human-readable) ----
json_path = os.path.join(CALIB_DIR, 'stereo_params.json')
stereo_params = {
    'image_size': list(frame_size),
    'baseline_m': float(np.linalg.norm(T)),
    'rms_stereo': float(ret_stereo),
    'K_L': K_L_s.tolist(),
    'dist_L': dist_L_s.tolist(),
    'K_R': K_R_s.tolist(),
    'dist_R': dist_R_s.tolist(),
    'R': R.tolist(),
    'T': T.tolist(),
    'Q': Q.tolist(),
    'ROI1': list(ROI1),
    'ROI2': list(ROI2),
}
with open(json_path, 'w') as f:
    json.dump(stereo_params, f, indent=2)
print(f'Saved stereo params to: {json_path}')

## Step 9: Loading and Using the Maps

In your production application, you **skip calibration entirely** — just load the maps:

In [ ]:
# ---- Load saved maps ----
stereo_data = np.load(npz_path)

map1_L_loaded = stereo_data['map1_L']
map2_L_loaded = stereo_data['map2_L']
map1_R_loaded = stereo_data['map1_R']
map2_R_loaded = stereo_data['map2_R']
Q_loaded      = stereo_data['Q']

print('Loaded stereo maps:')
print(f'  map1_L: {map1_L_loaded.shape}')
print(f'  Q: {Q_loaded.shape}')
print()

# ---- Apply to a frame ----
def apply_stereo_rectification(left_raw, right_raw,
                                map1_l, map2_l, map1_r, map2_r):
    """
    Rectify a stereo image pair. Call this every frame.
    This is the core runtime operation.
    """
    left_rect  = cv2.remap(left_raw,  map1_l, map2_l, cv2.INTER_LINEAR)
    right_rect = cv2.remap(right_raw, map1_r, map2_r, cv2.INTER_LINEAR)
    return left_rect, right_rect

# Test with synthetic images
left_raw  = np.random.randint(50, 200, (IMG_H, IMG_W, 3), dtype=np.uint8)
right_raw = np.random.randint(50, 200, (IMG_H, IMG_W, 3), dtype=np.uint8)

left_rect, right_rect = apply_stereo_rectification(
    left_raw, right_raw,
    map1_L_loaded, map2_L_loaded,
    map1_R_loaded, map2_R_loaded
)

print(f'Input shape:  {left_raw.shape}')
print(f'Output shape: {left_rect.shape}')
print('Rectification applied successfully.')

## Full Pipeline Summary

Here's the complete stereo vision workflow in production:

In [ ]:
# Two scripts in scripts/ cover this full pipeline:
#
# ── ONE-TIME SETUP ────────────────────────────────────────────────────────────
#   scripts/stereo_camera_calibration.py
#   Reads matching chessboard image pairs from two folders, runs the full
#   8-step pipeline, and saves stereo_maps.npz.
#
#   Usage:
#     python scripts/stereo_camera_calibration.py \
#       --left-dir images/left --right-dir images/right \
#       --cols 9 --rows 6 --square 0.025
#
# ── REAL-TIME DEPTH ───────────────────────────────────────────────────────────
#   scripts/stereo_depth_live.py
#   Loads stereo_maps.npz, opens two cameras, rectifies + disparity every frame.
#
#   Usage:
#     python scripts/stereo_depth_live.py --left 0 --right 1
#
#   Controls: Q=quit  D=cycle display (left+depth / rectified pair / depth only)
#
# Both scripts support --help for full argument list.

print("See scripts/stereo_camera_calibration.py and scripts/stereo_depth_live.py")

## Recap

| Step | Function | Key Outputs |
|---|---|---|
| Mono calibrate L | `calibrateCamera` | K_L, dist_L |
| Mono calibrate R | `calibrateCamera` | K_R, dist_R |
| Stereo calibrate | `stereoCalibrate` | R, T, E, F |
| Stereo rectify | `stereoRectify` | R1, R2, P1, P2, **Q**, ROI1, ROI2 |
| Build maps | `initUndistortRectifyMap` | map1_L/R, map2_L/R |
| Runtime | `remap` each frame | Rectified image pair |
| Depth | `StereoSGBM` + `reprojectImageTo3D` | Disparity map + 3D point cloud |

| Parameter | Value |
|---|---|
| `CALIB_FIX_INTRINSIC` | Fix K, dist during stereoCalibrate (recommended) |
| `alpha=0` in rectify | Crop to valid region (no black borders) |
| `CV_32FC1` in maps | Single-channel float maps (required by remap) |

**Part 6 Complete!** You now understand stereo vision theory and calibration.  
**Next:** Part 7 — Deep Learning approaches for markerless 6D pose estimation.

---
## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Compute depth at a specific pixel
# ============================================================
# Using the Q_loaded matrix:
# At pixel (320, 240) with disparity 35 pixels, what is the 3D position?
# Compute X, Y, Z in meters and report the distance from the camera.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# u, v, d = 320, 240, 35.0
# uvd1 = np.array([u, v, d, 1.0])
# XYZW = Q_loaded @ uvd1
# X, Y, Z = XYZW[:3] / XYZW[3]
# dist = np.sqrt(X**2 + Y**2 + Z**2)
# print(f'3D position: X={X:.3f}m, Y={Y:.3f}m, Z={Z:.3f}m')
# print(f'Distance from camera: {dist:.3f}m = {dist*100:.1f}cm')

In [ ]:
# ============================================================
# EXERCISE 2: Validate epipolar alignment
# ============================================================
# Take 5 points from the left calibration image pair.
# Use the Fundamental Matrix F to compute the epipolar line in the right image.
# Check if the corresponding right image point lies on the epipolar line.
# F is available as the output of stereoCalibrate above.
# Hint: epipolar line for point p_L is l = F @ p_L_homogeneous
#       a point p_R is on line l=(a,b,c) if a*xR + b*yR + c ≈ 0
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# # Take 5 point pairs from first calibration pair
# pts_L_test = imgpts_L[0].reshape(-1, 2)[:5]  # (5, 2)
# pts_R_test = imgpts_R[0].reshape(-1, 2)[:5]
# 
# print('Epipolar validation (lower = more aligned):')
# for (uL, vL), (uR, vR) in zip(pts_L_test, pts_R_test):
#     pL = np.array([uL, vL, 1.0])
#     pR = np.array([uR, vR, 1.0])
#     l = F @ pL              # epipolar line in right image
#     residual = abs(pR @ l) / np.sqrt(l[0]**2 + l[1]**2)  # point-to-line distance
#     print(f'  pL=({uL:.0f},{vL:.0f}) → epi_residual={residual:.4f} px')

In [ ]:
# ============================================================
# EXERCISE 3: Compare alpha=0 vs alpha=1 rectification
# ============================================================
# Run stereoRectify twice: once with alpha=0 and once with alpha=1.
# Compare ROI1 areas and describe the trade-off in your own words.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# for alpha in [0, 1]:
#     R1a, R2a, P1a, P2a, Qa, roi1a, roi2a = cv2.stereoRectify(
#         K_L_s, dist_L_s, K_R_s, dist_R_s,
#         frame_size, R, T, alpha=alpha)
#     x,y,w,h = roi1a
#     area_pct = (w*h) / (IMG_W*IMG_H) * 100
#     print(f'alpha={alpha}: ROI1={roi1a}, valid area={area_pct:.1f}%')
# # alpha=0: smaller valid area, no black borders, more useful pixels for stereo matching
# # alpha=1: full image kept, black borders at edges from warp, useful to see all content